# EJERCICIO 3: SELECCION DE VARIABLES MEDIANTE METODOS DE FILTRO

### cargo el ultimo dataset con las variables creadas a partir de los metodos manuales y automaticos

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import VarianceThreshold
import matplotlib.pyplot as plt
import seaborn as sns

df_ej3=pd.read_csv('dataset_ej2.csv')

df_ej3.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3429 entries, 0 to 3428
Data columns (total 68 columns):
 #   Column                                                                                                 Non-Null Count  Dtype  
---  ------                                                                                                 --------------  -----  
 0   country                                                                                                3429 non-null   object 
 1   categoria_expectativa_vida                                                                             3429 non-null   object 
 2   gdp_per_capita_current_us$                                                                             3429 non-null   float64
 3   government_integrity                                                                                   3429 non-null   float64
 4   energy_use_kg_of_oil_equivalent_per_capita                                                      

recordar que este dataset rancio no esta escalado y tiene 0 nulos.... pero antes voy a leer las consignas por las dudas

## 3.1: Filtro basado en varianza y correlacion

- **Filtro de varianza casi nula(VarianceThreshold)**: Pongo lo que entendi... basicamente es un metodo no supervisado(no le pasamos la y,trabaja a ciegas). si el 99% de los paises tuvieran el mismo valor en una columna(por ejemplo que todos tengan acceso a la electricidad), esa variable tendria varianza casi cero, no aportaria nada para distinguir un pais, por lo que se filtra.
- **Filtro por alta correlacion**: ataca la multicolinealidad (redundancia), en el ejercicio 2 creamos variables que basicamente pueden ser clones. Pone un limite de correlacion y quita las que lo superan, bah... una de ellas. Deja el dataset mas limpio

In [2]:
#POR VARIANZA
#guiandome nuevamente con los codigos de la teoria
target = 'gdp_per_capita_current_us$'
cols_texto = ['country', 'categoria_expectativa_vida']

#dividimos aislando las numericas
X = df_ej3.drop(columns=cols_texto + [target])
y = df_ej3[target]

#escalo entre 0 y 1, en la teoria dice que se aplica sobre datos escalados
scaler = MinMaxScaler()
X_escalada = pd.DataFrame(scaler.fit_transform(X), columns=X.columns, index=X.index)
#si uso el standarsclaer la varianza daria 1, esto entiendo que haria que el VarianceThreshold no sirva.
#como llevamos todo entre 0 y 1 , si hay una columna casi llena de 0, el filtro la detecta 

# Umbral 0.01: Elimina si el 99% de los valores de la columna son idénticos
umbral_var = 0.01
filtro_var = VarianceThreshold(threshold=umbral_var)
filtro_var.fit(X_escalada)

#nos quedamos solo con las columnas que superaron el umbral
cols_sobrevivientes_var = X_escalada.columns[filtro_var.get_support()]
X_var = X_escalada[cols_sobrevivientes_var]

print(f"columnas base (automáticas + manuales): {len(X_escalada.columns)}")
print(f"sobrevivientes al filtro de varianza: {len(X_var.columns)}")

#POR CORRELACION
#para eliminar redundancia o multicolinealidad (lo que hicimos a pata en el 2)
matriz_corr = X_var.corr().abs()
triangulo_sup = matriz_corr.where(np.triu(np.ones(matriz_corr.shape), k=1).astype(bool))

#elegimos un ubral de 0.85 como esta en la teoria
umbral_corr = 0.85
columnas_a_borrar_corr = [col for col in triangulo_sup.columns if any(triangulo_sup[col] > umbral_corr)]
#eliminamos redundancia
X_final_3_1 = X_var.drop(columns=columnas_a_borrar_corr)
print(f"columnas eliminadas por alta correlación (>0.85): {len(columnas_a_borrar_corr)}")
print(f"columnas Sobrevivientes: {len(X_final_3_1.columns)}")



columnas base (automáticas + manuales): 65
sobrevivientes al filtro de varianza: 37
columnas eliminadas por alta correlación (>0.85): 26
columnas Sobrevivientes: 11


Lo del traignulo superior tambien estaba en la teoria, tapa la mitad de abajo y la diagonal y obliga a mirar cada para de variables una vez. Lo copie por las dudas pero entiendo que aca es donde se filtra cual de las dos eliminar. Es decir, el triangulo pone NaN la parte inferior de la matriz, luego si tenemos que dos variables tienen corr=0.9, solo se muestra en la parte de arriba, en la de abajo sale NaN. Entonces cuando hacemos el any() , revisa la primer columna de este par y ve el NaN, luego se encuentra que la otra columna tiene 0.9 con la anterior, y la borra, entonces la primer columna se salvo porque estaba primero en la matriz y su correlacion salia abajo. Medio engorroso pero se entiende

Preguntas para cuando nos tomen: 
- Siempre se aplican ambos metodos?
- El orden importa? es decir, primero varianza luego correlacion?


reconstruyo el df

In [3]:
df_ej3_1= pd.concat([df_ej3[cols_texto], X_final_3_1, y], axis=1)

## 3.2 Seleccion univariada supervisada

- **F-test**: evalua la relacion lineal entre cada predictoria y el target
    - alpha=0.05 (nivel de significancia)
    - H0: el coef de correlacion es 0, no hay relacion entre la caracteristica y el PBI (si p-valor > 0.05, no podemos rechazar H0)
    - H1: el coef de correlacion es distinto de 0, hay relacion lineal significativa (p-valor < 0.05)



In [4]:
from sklearn.feature_selection import SelectKBest, f_regression, mutual_info_regression
#usamos las variables numericas que sobrevivieron al ejercicio anterior
X_3_1= X_final_3_1

#nos quedamos con las 5 mejores caract
k_mejores=5

#F-Test
selector_f = SelectKBest(score_func=f_regression, k=k_mejores)
selector_f.fit(X_3_1, y)
df_scores_f = pd.DataFrame({
    'Variable': X_3_1.columns,
    'F-Score (relacion lineal)': selector_f.scores_,
    'p-valor': selector_f.pvalues_
}).sort_values(by='F-Score (relacion lineal)', ascending=False)

print("El top 5 segun el F-test es: ")
print(df_scores_f.head(k_mejores).to_string(index=False))
print("\n" + "="*70 + "\n")



El top 5 segun el F-test es: 
                                           Variable  F-Score (relacion lineal)       p-valor
                               government_integrity                5536.124395  0.000000e+00
bin_auto_energy_use_kg_of_oil_equivalent_per_capita                2844.636112  0.000000e+00
         energy_use_kg_of_oil_equivalent_per_capita                2745.586157  0.000000e+00
               life_expectancy_at_birth_total_years                2146.511107  0.000000e+00
                     agriculture_value_added_of_gdp                1271.562890 3.811877e-237




En este caso, como era de esperar, la variable 'govermente_integrity' gano, pq el metodo es similar al de correlacion de pearson, y esta variable era la q mas se correlacionaba.

- **Informacion mutua**: evalua cualquier tipo de informacion (lineal o no lineal). Mide cuanta incertidumbre (entropia) sobre el PBI desaparece cuando conocemos la variable X. Si la informacion mutua es 0, las variables son independientes

In [5]:
selector_mi = SelectKBest(score_func=mutual_info_regression, k=k_mejores)
selector_mi.fit(X_3_1, y)
df_scores_mi = pd.DataFrame({
    'Variable': X_3_1.columns,
    'Mutual Information Score': selector_mi.scores_
}).sort_values(by='Mutual Information Score', ascending=False)

print("Top 5 variables segun el metodo de INFORMACION MUTUA: ")
print(df_scores_mi.head(k_mejores).to_string(index=False))



Top 5 variables segun el metodo de INFORMACION MUTUA: 
                                                          Variable  Mutual Information Score
                                    agriculture_value_added_of_gdp                  0.984635
ratio_auto_agriculture_value_added_of_gdp_div_government_integrity                  0.981179
                        energy_use_kg_of_oil_equivalent_per_capita                  0.967851
                              life_expectancy_at_birth_total_years                  0.862545
               bin_auto_energy_use_kg_of_oil_equivalent_per_capita                  0.730410


### Conclusion

Estas 4 se repiten en relacion al F-tes:
- energy_use_kg_of_oil_equivalent_per_capita
- bin_auto_energy_use...
- life_expectancy_at_birth_total_years
- agriculture_value_added_of_gdp

Que estas queden en ambos metodos demuestra que son los mejores predictores. Tienen mucha relacion con el PBI sin importar como lo mires.

- Lo novedoso: la variable **agriculture_value_added_of_gdp** pego un salto, quedo ultimo en el F-test y primero en el MI, esto entiendo que se debe a que la relacion entre el peso del agro y el PBI es una curva... los paises pobres dependen mucho del agro pero a medida que se modernizan el porcentaje cae, fue correcta elegirla.

In [6]:
#guardamos los df finales con las 5 mejores de cada metodo
X_final_ftest = X_3_1[X_3_1.columns[selector_f.get_support()]]
X_final_mi = X_3_1[X_3_1.columns[selector_mi.get_support()]]

## 3.3 Comparacion y validacion de modelos

Usamos regresion lineal y medimos el R^2. Es similar al ejercicio 6 del tp anterior 

In [7]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PowerTransformer, StandardScaler

# Definimos los 4 subconjuntos a comparar
subconjuntos = {
    "1. Modelo FILTRADO BASE (11 variables de 3.1)": X_3_1,
    "2. Modelo F-TEST (5 mejores lineales de 3.2)": X_final_ftest,
    "3. Modelo INFO MUTUA (5 mejores no lineales 3.2)": X_final_mi,
}

# Configuración de Validación Cruzada (5 folds estables)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
resultados_cv = []

for nombre, X_subset in subconjuntos.items():
    # Pipeline idéntico para evitar fugas de información
    pipeline = Pipeline(
        [
            ("power_transform", PowerTransformer(method="yeo-johnson")),
            ("scaler", StandardScaler()),
            ("modelo", LinearRegression()),
        ]
    )

    scores = cross_val_score(pipeline, X_subset, y, cv=kf, scoring="r2")

    resultados_cv.append(
        {
            "Subconjunto Evaluado": nombre,
            "Cant. Variables": X_subset.shape[1],
            "R2 Promedio": np.mean(scores),
            "R2 Std": np.std(scores),
        }
    )

# Visualización comparativa tabular
df_resumen_filtros = pd.DataFrame(resultados_cv).sort_values(
    by="R2 Promedio", ascending=False
)
print("=" * 80)
print("VALIDACIÓN CRUZADA (5-FOLD CV) - COMPARATIVA DE MÉTODOS DE FILTRO")
print("=" * 80)
print(df_resumen_filtros.to_string(index=False))

VALIDACIÓN CRUZADA (5-FOLD CV) - COMPARATIVA DE MÉTODOS DE FILTRO
                            Subconjunto Evaluado  Cant. Variables  R2 Promedio   R2 Std
   1. Modelo FILTRADO BASE (11 variables de 3.1)               11     0.786549 0.009860
    2. Modelo F-TEST (5 mejores lineales de 3.2)                5     0.652006 0.016317
3. Modelo INFO MUTUA (5 mejores no lineales 3.2)                5     0.627662 0.014183


Bueno si hice todo bien, gano el modelo base y con una diferencia de 14 puntacos.
- Creo que en los otros metodos perdieron por la seleccion univariada... tal vez eliminamos variables que por si sola no aportaban nada pero daban como un contexto diferente para valorar el PIB per capita de un pais
- Usando todas las variables, esto explota! Esto se produce a que al usar todas las columnas, y hacer la regresion de minimos cuadrados $(X^T X)^{-1}$ hace que el det de casi 0

Las 11 variables logran explicar casi e 79% de la varianza del PBI. cuak

# EJERCICIO 4: METODOS WRAPPER Y EMBEBIDOS PARA SELECCION DE VARIABLES

## 4.1 Seleccion secuencial hacia adelante y hacia atras

- **Forward Selection (hacia adelante)**:  Empieza con 0 variables. Prueba las 11 por separado, se queda con la que da mejor R2. Luego prueba sumarle una segunda, y se queda con el mejor dúo. Así sucesivamente hasta llegar al cirterio de corte.
- **Backward Selection (hacia atras)**: Empieza con el equipo completo (11 variables). Prueba sacar a una, y si el R2 no empeora casi nada, la echa. Sigue echando a las peores hasta que queden las que queremos segun algun criterio de corte.



Voy a seguir usando regresion, no uso la libreria de la consiga (sklearn.ensamle), se que es para random forest pero si antes hicimos regresion lo voy a seguir comparando con eso para ver si realmente estos metodos son mejores. Manzanas con manzanas

In [13]:
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PowerTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
import pandas as pd
import numpy as np

#el q uso para comparar
modelo_juez = Pipeline([
    ('power_transform', PowerTransformer(method='yeo-johnson')),
    ('scaler', StandardScaler()),
    ('modelo', LinearRegression())
])
k_mejores = 5

#Hacia adelante, en la teoria no habia narnia de esto, hay que estudiar bien q hace
modelo_forward = SequentialFeatureSelector(
    estimator=modelo_juez, 
    n_features_to_select=k_mejores, 
    direction='forward',
    scoring='r2',
    cv=kf
)
modelo_forward.fit(X_3_1, y)
cols_forward = X_3_1.columns[modelo_forward.get_support()]

#Hacia atras
modelo_backward = SequentialFeatureSelector(
    estimator=modelo_juez, 
    n_features_to_select=k_mejores, 
    direction='backward',
    scoring='r2',
    cv=kf
)
modelo_backward.fit(X_3_1, y)
cols_backward = X_3_1.columns[modelo_backward.get_support()]

#comparamos
score_forward = cross_val_score(modelo_juez, X_3_1[cols_forward], y, cv=5, scoring='r2').mean()
score_backward = cross_val_score(modelo_juez, X_3_1[cols_backward], y, cv=5, scoring='r2').mean()

#RECORDEMOS QUE EL MODELO BASE DABA  0.7865
print("R2 del modelo con las 11 variables del anterior: 0.7865")
print("\nR2 del mejor univariado: 0.65 ")
print(""*15)
print(f"R2 del modelo_forward: {score_forward}")
print("=" * 70)
print("\n1. FORWARD SELECTION (5 variables):")
for i, col in enumerate(cols_forward, 1):
    print(f"  {i}. {col}")
print("="*15)
print(f"R2 del modelo_backwar: {score_backward}")
print("=" * 70)
print("\n2. BACKWARD ELIMINATION (5 variables):")
for i, col in enumerate(cols_backward, 1):
    print(f"  {i}. {col}")

# Intersección entre ambos enfoques
coincidentes_wrapper = set(cols_forward).intersection(set(cols_backward))
print(f"\nVariables coincidentes en ambos wrappers: {len(coincidentes_wrapper)} de 5")
for col in coincidentes_wrapper:
    print(f"  * {col}")


R2 del modelo con las 11 variables del anterior: 0.7865

R2 del mejor univariado: 0.65 

R2 del modelo_forward: 0.7428166689345617

1. FORWARD SELECTION (5 variables):
  1. energy_use_kg_of_oil_equivalent_per_capita
  2. life_expectancy_at_birth_total_years
  3. agriculture_value_added_of_gdp
  4. ratio_auto_energy_use_kg_of_oil_equivalent_per_capita_div_government_integrity
  5. ratio_auto_agriculture_value_added_of_gdp_div_government_integrity
R2 del modelo_backwar: 0.7552594596152418

2. BACKWARD ELIMINATION (5 variables):
  1. government_integrity
  2. energy_use_kg_of_oil_equivalent_per_capita
  3. government_integrity infant_mortality_rate_per_1_000_live_births
  4. ratio_auto_agriculture_value_added_of_gdp_div_government_integrity
  5. ratio_auto_infant_mortality_rate_per_1_000_live_births_div_agriculture_value_added_of_gdp

Variables coincidentes en ambos wrappers: 2 de 5
  * ratio_auto_agriculture_value_added_of_gdp_div_government_integrity
  * energy_use_kg_of_oil_equivalent_

### Conclusion

En este caso Wrapper > Filtros
- Esto pasa porque wrapper no elige los 5 de manera individual como los metodos anteriores, busca la combinacion de los 5 mejores, los que mejor se complementen.
- Nuestro modelo base requeria de 11 caracteristicas, el Backward con 5 caracteristicas solo pierde 3 puntos, es una ganancia genial, es un modelo mucho mas simple y mas rapido

## 4.2 Metodos embebidos con regularizacion

Metodos embebidos: no testean a ciegas como los filtros ni corren mil veces como los wrapper, ahcen la seleccion al mismo tiempo que aprenden.
- Regularización L1 (Lasso): Es un algoritmo que penaliza la complejidad Mientras entrena, si nota que una variable no aporta casi nada, literalmente encoge el peso (coeficiente) de esa variable hasta llegar exactamente a 0.0. Al quedar en cero, la variable queda eliminada del modelo ("esparsidad").
- Elastic Net: Es una mezcla entre Lasso (L1) y Ridge (L2). Lasso suele ser muy agresivo y, si hay dos variables similares, borra una al azar. Elastic Net reparte el castigo de forma mas chill, siendo mejor para datos con multicolinealidad.

In [9]:
from sklearn.linear_model import LassoCV, ElasticNetCV
from sklearn.feature_selection import SelectFromModel
from sklearn.preprocessing import StandardScaler

#escalar para lasso y elastic es el obligatorio, quise hacer con mixmaxscaler pero daba cualquier cosa
#a estos modelos les gusta la media 0 segun un amigo, pero no me acuerdo por que
scaler = StandardScaler()
X_escalada = pd.DataFrame(scaler.fit_transform(X_3_1), columns=X_3_1.columns, index=X_3_1.index)

#LASSO
modelo_lasso = LassoCV(cv=5, random_state=42)
modelo_lasso.fit(X_escalada, y)
#SelectFromModel extrae solo las columnas a las que Lasso NO les puso coeficiente 0.
selector_lasso = SelectFromModel(modelo_lasso, prefit=True)
cols_lasso = X_escalada.columns[selector_lasso.get_support()]

print(f"sobrevivientes a LASSO= ({len(cols_lasso)} de 11 variables):")
for col in cols_lasso:
    print(f"   - {col}")

#ELASTIC NET (L1+L2)
#l1_ratio=0.5 significa que aplica 50% de agresividad (lasso) y 50% de suavidad (Ridge)
modelo_elastic = ElasticNetCV(l1_ratio=0.5, cv=5, random_state=42)
modelo_elastic.fit(X_escalada, y)
selector_elastic = SelectFromModel(modelo_elastic, prefit=True)
cols_elastic = X_escalada.columns[selector_elastic.get_support()]

print(f"sobrevivientes a ELASTIC NET=({len(cols_elastic)} de 11 variables):")
for col in cols_elastic:
    print(f"   - {col}")

#Comparamos con los anteriores
#usamos el modelo_juez con yeo-johnson del 4.1
score_lasso = cross_val_score(modelo_juez, X_3_1[cols_lasso], y, cv=5, scoring='r2').mean()
score_elastic = cross_val_score(modelo_juez, X_3_1[cols_elastic], y, cv=5, scoring='r2').mean()
print("\n" + "="*50)
print("COMPARATIVA FINAL DE R2 ")
print(f"modelo base (11 var completas): 0.7865")
print(f"mejor del Wrapper (5 var):  0.7552")
print("-" * 30)
print(f"R2 de Lasso ({len(cols_lasso)} var):       {score_lasso:.4f}")
print(f"R2 de ElasticNet ({len(cols_elastic)} var):  {score_elastic:.4f}")



sobrevivientes a LASSO= (8 de 11 variables):
   - government_integrity
   - energy_use_kg_of_oil_equivalent_per_capita
   - life_expectancy_at_birth_total_years
   - services_value_added_of_gdp
   - tiene_dato_gini
   - government_integrity infant_mortality_rate_per_1_000_live_births
   - ratio_auto_agriculture_value_added_of_gdp_div_government_integrity
   - ratio_auto_infant_mortality_rate_per_1_000_live_births_div_agriculture_value_added_of_gdp
sobrevivientes a ELASTIC NET=(4 de 11 variables):
   - government_integrity
   - energy_use_kg_of_oil_equivalent_per_capita
   - life_expectancy_at_birth_total_years
   - bin_auto_energy_use_kg_of_oil_equivalent_per_capita

COMPARATIVA FINAL DE R2 
modelo base (11 var completas): 0.7865
mejor del Wrapper (5 var):  0.7552
------------------------------
R2 de Lasso (8 var):       0.7568
R2 de ElasticNet (4 var):  0.6399


- Lasso le gano un poco al wrapper pero con 3 variables mas.... sigue siendo mejor el de backward selection.

- Elastic net quito muchas variables pero su R2 se desplomo, 4 variables son insuficientes

## 4.3 Comparaciones mas pros

Por las dudas lo ahcemos asi cumplo la consigna

In [10]:
from sklearn.model_selection import cross_validate ,GridSearchCV
from sklearn.linear_model import Ridge

equipos_competidores = {
    "1. BASELINE (Todas las 11 variables)": X_3_1,
    "2. FILTRO Univariado F-Test (5 var)": X_final_ftest,
    "3. WRAPPER Backward Selection (5 var)": X_3_1[cols_backward],
    "4. EMBEBIDO Lasso (Selección automática)": X_3_1[cols_lasso]
}
resultados_finales = []

#CV
for nombre, X_sub in equipos_competidores.items():
    # cross_validate nos permite pedir varias metricas al mismo tiempo
    metricas = ['r2', 'neg_mean_squared_error']
    cv_resultados = cross_validate(
        estimator=modelo_juez, #el del 4.1
        X=X_sub, 
        y=y, 
        cv=5, 
        scoring=metricas
    )
    
    #promediamos el R2
    r2_promedio = np.mean(cv_resultados['test_r2'])

    # Scikit-Learn devuelve el MSE en negativo (porque siempre busca maximizar). 
    # Le ponemos un signo "menos" adelante para volverlo positivo.
    mse_promedio = -np.mean(cv_resultados['test_neg_mean_squared_error'])
    
    resultados_finales.append({
        "Método de Selección": nombre,
        "Variables": len(X_sub.columns),
        "R2": round(r2_promedio, 4),
        "MSE (Más cerca de 0 es MEJOR)": round(mse_promedio, 2)
    })

#tabla final
df_marcador = pd.DataFrame(resultados_finales)
print(df_marcador.to_string(index=False))


                     Método de Selección  Variables     R2  MSE (Más cerca de 0 es MEJOR)
    1. BASELINE (Todas las 11 variables)         11 0.7570                    92684480.67
     2. FILTRO Univariado F-Test (5 var)          5 0.6395                   136889991.65
   3. WRAPPER Backward Selection (5 var)          5 0.7553                    92374809.81
4. EMBEBIDO Lasso (Selección automática)          8 0.7568                    91405688.80


- **Baseline (11 variables)**: Se equivoca por 92.6 millones.
- **Wrapper Backward (5 variables)**: Se equivoca por 92.3 millones. El error bajo papaaaaa
- **Lasso (8 variables)**: Se equivoca por 91.4 millones. Bajo mas pero tiene 3 variables mas que el anterior 

el mas top es wrapper Backward

In [11]:
#Usamos Gridsearch para optimizar el ganador
# Armamos un pipeline nuevo usando Ridge (Regresión Lineal con penalización L2)
pipeline_ridge = Pipeline([
    ('power_transform', PowerTransformer(method='yeo-johnson')),
    ('modelo_ridge', Ridge())
])
# Le decimos que pruebe estos distintos valores de alpha
grilla_parametros = {
    'modelo_ridge__alpha': [0.1, 1.0, 10.0, 100.0]
}

# GridSearchCV hace CV probando todas las configuraciones
optimizador = GridSearchCV(
    estimator=pipeline_ridge,
    param_grid=grilla_parametros,
    cv=5,
    scoring='r2'
)

#entrenamos solo con el subconjunto ganador del Wrapper Backward
optimizador.fit(X_3_1[cols_backward], y)
print(f"Mejor configuración encontrada: {optimizador.best_params_}")
print(f"R2 superoptimizado del Wrapper: {optimizador.best_score_:.4f}")

Mejor configuración encontrada: {'modelo_ridge__alpha': 10.0}
R2 superoptimizado del Wrapper: 0.7580


Bueno el salto de calidad claramente no estuvo al final, si no al princio... se verifica que el 80% esta en el analisis previo, no sirvio de mucho pero bueno.

# Ejercicio 5

## 5.1

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

print("=" * 80)
print("5.1 REDUCCIÓN DE DIMENSIONALIDAD LINEAL: PCA")
print("=" * 80)

#matriz de entrada: Tomamos las 65 variables generadas (o X_todas)
cols_texto = ["country", "categoria_expectativa_vida"]
target = "gdp_per_capita_current_us$"

df_ej5=pd.read_csv('dataset_ej2_var_imp.csv')
df_ej5 = df_ej5.drop(columns=['Unnamed: 0'])

#predictoras numericas, borrando la categoricas
X_features = df_ej5.drop(
    columns=[c for c in cols_texto + [target] if c in df_ej5.columns]
)
y_target = df_ej5[target]
y_log_target = np.log1p(y_target)

#estandarizacion  PCA
scaler_pca = StandardScaler()
X_scaled = scaler_pca.fit_transform(X_features)

print(f"Dimensiones de entrada: {X_scaled.shape[0]} muestras x {X_scaled.shape[1]} variables")

#ajuste de PCA completo
pca_full = PCA(random_state=42)
pca_full.fit(X_scaled)

var_individual = pca_full.explained_variance_ratio_
var_acumulada = np.cumsum(var_individual)

#cortes optimos (80% y 95%)
k_80 = np.argmax(var_acumulada >= 0.80) + 1
k_95 = np.argmax(var_acumulada >= 0.95) + 1

print(f"\nNúmero de componentes para explicar >= 80% de varianza: {k_80} (de {X_scaled.shape[1]})")
print(f"Número de componentes para explicar >= 95% de varianza: {k_95} (de {X_scaled.shape[1]})")
print(f"Varianza explicada por PC1: {var_individual[0]*100:.2f}%")
print(f"Varianza explicada por PC2: {var_individual[1]*100:.2f}%")
print(f"Varianza explicada conjunta (PC1 + PC2): {(var_individual[0] + var_individual[1])*100:.2f}%")


fig, ax1 = plt.subplots(figsize=(10, 5), dpi=110)
comps = np.arange(1, len(var_individual) + 1)
ax1.bar(comps[:25], var_individual[:25] * 100, alpha=0.5, color="royalblue", label="Varianza Individual (%)")
ax1.plot(comps[:25], var_acumulada[:25] * 100, marker="o", color="crimson", linewidth=2, label="Varianza Acumulada (%)")
ax1.axhline(80, color="green", linestyle="--", linewidth=1.2, label=f"Umbral 80% ({k_80} comps)")
ax1.axhline(95, color="purple", linestyle=":", linewidth=1.2, label=f"Umbral 95% ({k_95} comps)")
ax1.set_xlabel("Número de Componentes Principales")
ax1.set_ylabel("Porcentaje de Varianza Explicada (%)")
ax1.set_title("Scree Plot: Análisis de Varianza Explicada (Primeros 25 componentes)")
ax1.set_xticks(range(1, 26))
ax1.grid(True, linestyle="--", alpha=0.4)
ax1.legend(loc="center right")

plt.tight_layout()
plt.show()

#2D (PC1 vs PC2) coloreada por PIB per cápita
pca_2d = PCA(n_components=2, random_state=42)
X_pca_2d = pca_2d.fit_transform(X_scaled)

df_pca_plot = pd.DataFrame({
    "PC1": X_pca_2d[:, 0],
    "PC2": X_pca_2d[:, 1],
    "PIB_per_capita": y_target,
    "log_PIB": y_log_target
})

plt.figure(figsize=(9, 6), dpi=110)
scatter = plt.scatter(
    df_pca_plot["PC1"],
    df_pca_plot["PC2"],
    c=df_pca_plot["log_PIB"],
    cmap="viridis",
    alpha=0.6,
    s=25,
    edgecolors="none"
)

cbar = plt.colorbar(scatter)
cbar.set_label("Log(PIB per cápita USD)", fontsize=11)
plt.title(
    f"Proyección Espacial PCA (PC1 vs PC2)\n"
    f"PC1 ({var_individual[0]*100:.1f}%) | PC2 ({var_individual[1]*100:.1f}%) - Varianza Total: {(var_individual[0]+var_individual[1])*100:.1f}%",
    fontsize=12
)
plt.xlabel(f"PC1 ({var_individual[0]*100:.1f}% varianza explicada)")
plt.ylabel(f"PC2 ({var_individual[1]*100:.1f}% varianza explicada)")
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

#Variables con mayor peso en PC1 y PC2)
loadings = pd.DataFrame(
    pca_2d.components_.T,
    columns=["Load_PC1", "Load_PC2"],
    index=X_features.columns
)

print("\n" + "=" * 70)
print("TOP 5 VARIABLES CON MAYOR INFLUENCIA ABSOLUTA EN PC1:")
print("=" * 70)
print(loadings.reindex(loadings["Load_PC1"].abs().sort_values(ascending=False).index).head(5)[["Load_PC1"]].round(4))

print("\n" + "=" * 70)
print("TOP 5 VARIABLES CON MAYOR INFLUENCIA ABSOLUTA EN PC2:")
print("=" * 70)
print(loadings.reindex(loadings["Load_PC2"].abs().sort_values(ascending=False).index).head(5)[["Load_PC2"]].round(4))

5.1 REDUCCIÓN DE DIMENSIONALIDAD LINEAL: PCA


KeyError: "['Unnamed: 0'] not found in axis"

Como bien veniamos sabiendo, los datos que mauor influencia tienen en la varianza para una de las variables, tiene que ver con la faltante de datos, debido a que esto no es prioridad en estos paises (explicado en la variable tiene_dato_gini), y aquello relacionado con la expectativa de vida, ya que por la falta de infraestructura basica, se disminuye,

Realizar esto permite achicar drásticamente la cantidad de variables de entrada, evitando sufrir la "maldición de la dimensionalidad" y reduciendo el riesgo overfittear, y podemos tmb conservar la mayor cantidad de información posible.  
Como metodo de preprocesamiento, usar PCA transforma variables, originalmente correlacionadas, en un nuevo conjunto de componentes principales, las cuales son ortogonales!, eliminando el problema de la multicolinealidad.

## 5.2

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import umap.umap_ as umap 


#t-SNE
tsne = TSNE(n_components=2, random_state=42)
X_tsne = tsne.fit_transform(X_scaled)

#aplicar UMAP
umap_model = umap.UMAP(n_components=2, random_state=42)
X_umap = umap_model.fit_transform(X_scaled)
#crear el lienzo con 2 subgraficos lado a lado
fig = make_subplots(rows=1, cols=2, subplot_titles=('Proyección t-SNE', 'Proyección UMAP'))

#variables para no repetir codigo
paises = df_ej5['country']
pib = df_ej5['gdp_per_capita_current_us$']

#grafico t-SNE (Panel Izquierdo)
fig.add_trace(
    go.Scatter(
        x=X_tsne[:, 0], 
        y=X_tsne[:, 1],
        mode='markers',
        marker=dict(
            color=pib, 
            colorscale='Viridis', 
            opacity=0.7,
            colorbar=dict(title="PIB per cápita (US$)", x=0.46, thickness=15) # Barra color en el medio
        ),
        text=paises, #
        hovertemplate="<b>País: %{text}</b><br>PIB: US$ %{marker.color:,.0f}<extra></extra>"
    ),
    row=1, col=1
)

#grafico UMAP (Panel Derecho)
fig.add_trace(
    go.Scatter(
        x=X_umap[:, 0], 
        y=X_umap[:, 1],
        mode='markers',
        marker=dict(
            color=pib, 
            colorscale='Viridis', 
            opacity=0.7,
            colorbar=dict(title="PIB per cápita (US$)", x=1.02, thickness=15) # Barra color a la derecha
        ),
        text=paises,
        hovertemplate="<b>País: %{text}</b><br>PIB: US$ %{marker.color:,.0f}<extra></extra>"
    ),
    row=1, col=2
)


fig.update_layout(
    height=600, 
    width=1200, 
    showlegend=False,
    plot_bgcolor='white',
    hovermode='closest'
)

#replicar tus etiquetas de ejes y cuadrícula
fig.update_xaxes(title_text='Dimensión 1', showline=True, linecolor='lightgrey', gridcolor='whitesmoke', row=1, col=1)
fig.update_yaxes(title_text='Dimensión 2', showline=True, linecolor='lightgrey', gridcolor='whitesmoke', row=1, col=1)
fig.update_xaxes(title_text='Dimensión 1', showline=True, linecolor='lightgrey', gridcolor='whitesmoke', row=1, col=2)
fig.update_yaxes(title_text='Dimensión 2', showline=True, linecolor='lightgrey', gridcolor='whitesmoke', row=1, col=2)

fig.show()

d:\existencia\facu cosa\lab_2\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Estos 2 metodos nos permiten visualizar clusters de datos!
Podemos ver varios clusters pequeños, en los cuales encontramos paises con pib per capita similar! especificamente los pequeños, ya que se agrupan los paises "ricos" en distintas categorias, como veniamos viendo, la gran mayoria de los paises tienen pib bajo, por lo que esto nos permite hacer una mejor segmentacion a la vista

## 5.3

In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression

print("=" * 80)
print("5.3 AUTOCODIFICADORES PARA REDUCCIÓN DIMENSIONAL NO LINEAL")
print("=" * 80)


dim_original = X_scaled.shape[1]

#PCA (Reducción a 2 dimensiones)
pca = PCA(n_components=4, random_state=42)
X_pca = pca.fit_transform(X_scaled)
X_reconstructed_pca = pca.inverse_transform(X_pca)  # Devolvemos las 2D al espacio original

mse_pca = mean_squared_error(X_scaled, X_reconstructed_pca)

#No Lineal: Autocodificador con MLPRegressor
# Diseñamos el "cuello de botella": Entrada -> 16 -> 4 (Latente) -> 16 -> Salida
autoencoder = MLPRegressor(
    hidden_layer_sizes=(16, 4, 16), 
    activation='relu',
    solver='adam',
    max_iter=2000,
    random_state=42
)

# Entrenamos la red para que intente predecir su propia entrada (aprendizaje no supervisado)
autoencoder.fit(X_scaled, X_scaled)
X_reconstructed_ae = autoencoder.predict(X_scaled)

mse_ae = mean_squared_error(X_scaled, X_reconstructed_ae)

print("\nCALIDAD DE LA REPRESENTACIÓN (ERROR DE RECONSTRUCCION) ---")
print(f"MSE PCA (Lineal):          {mse_pca:.4f}")
print(f"MSE Autocodificador (Red): {mse_ae:.4f}")
print("*(Valores más cercanos a 0 indican que se perdió menos información al comprimir)*")

#Predecir el PIB per cápita (Rendimiento del modelo)
modelo_evaluador = LinearRegression()

#Baseline: Predecir usando todos los datos originales
modelo_evaluador.fit(X_scaled, y_target)
r2_original = r2_score(y_target, modelo_evaluador.predict(X_scaled))

#evaluar con datos reconstruidos por PCA
modelo_evaluador.fit(X_reconstructed_pca, y_target)
r2_pca = r2_score(y_target, modelo_evaluador.predict(X_reconstructed_pca))

#evaluar con datos reconstruidos por el Autocodificador
modelo_evaluador.fit(X_reconstructed_ae, y_target)
r2_ae = r2_score(y_target, modelo_evaluador.predict(X_reconstructed_ae))

print("\nRENDIMIENTO EN TAREAS POSTERIORES (R2 SCORE) ---")
print(f"R2 usando 100% variables originales: {r2_original:.4f}")
print(f"R2 usando compresión PCA (Lineal):   {r2_pca:.4f}")
print(f"R2 usando compresión Autoencoder:    {r2_ae:.4f}")

5.3 AUTOCODIFICADORES PARA REDUCCIÓN DIMENSIONAL NO LINEAL

--- 1. CALIDAD DE LA REPRESENTACIÓN (ERROR DE RECONSTRUCCIÓN) ---
MSE PCA (Lineal):          0.1451
MSE Autocodificador (Red): 0.0778
*(Valores más cercanos a 0 indican que se perdió menos información al comprimir)*

--- 2. RENDIMIENTO EN TAREAS POSTERIORES (R2 SCORE) ---
R2 usando 100% variables originales: 0.7325
R2 usando compresión PCA (Lineal):   0.6443
R2 usando compresión Autoencoder:    0.7919


El autoencoder logra distinguir relaciones no lineales, que son esperables de un sistema complejo como es un dataset economico por lo que nos permite conseguir mejores representaciones comprimidas, y mejores resultados que las otras representaciones!


#### 5.4

- Para la visualizacion, t-SNE y UMAP demostraron ser inmensamente superiores al PCA lineal. Con esto se logra mapear los datos a un espacio 2D preservando la estructura local (y global en el caso de UMAP), por lo que nos da una muy buena identificación visual de clústeres y relaciones complejas entre los países!!

- El Autocodificador es buenisimo!! Estas características "predichas" o "codificadas" en el espacio latente, nos da representaciones más efectivas y compactas para ser usadas como predictores en el modelo de regresión posterior, ya que la representacion es menos ruidosa que los datos origianles.  El PCA se limita a maximizar la varianza lineal, el Autocodificador conservó mejor la información subyacente. Safamos de la "maldición de la dimensionalidad" creando características artificiales que funcionaban bien, y asi, el modelo predictivo final generaliza mejor que si hubiera procesado los datos crudos.

# EJERCICIO 6

In [16]:
resumen_total = [
    {"Método": "11. Autocodificador (Espacio Latente)", "Dimensiones": 4, "R2 CV": 0.7919},
    {"Método": "2. Filtro Correlación (Base - 11 var)", "Dimensiones": 11, "R2 CV": 0.7865},
    {"Método": "7. Embebido Lasso", "Dimensiones": 8, "R2 CV": 0.7828},
    {"Método": "9. Wrapper Optimizado (Ridge)", "Dimensiones": 5, "R2 CV": 0.7767},
    {"Método": "6. Wrapper Backward", "Dimensiones": 5, "R2 CV": 0.7553},
    {"Método": "5. Wrapper Forward", "Dimensiones": 5, "R2 CV": 0.7428},
    {"Método": "10. Reducción Lineal PCA", "Dimensiones": 4, "R2 CV": 0.6443},
    {"Método": "8. Embebido ElasticNet", "Dimensiones": 4, "R2 CV": 0.6399},
    {"Método": "3. Filtro Univariado F-Test", "Dimensiones": 5, "R2 CV": 0.6395},
    {"Método": "4. Filtro Univariado Info Mutua", "Dimensiones": 5, "R2 CV": 0.6276},
]

import pandas as pd
df_resumen = pd.DataFrame(resumen_total)
# Lo mostramos ordenado por R2 descendente
df_resumen.sort_values(by="R2 CV", ascending=False).reset_index(drop=True)


,Método,Dimensiones,R2 CV
0,11. Autocodificador (Espacio Latente),4,0.7919
1,2. Filtro Correlación (Base - 11 var),11,0.7865
2,7. Embebido Lasso,8,0.7828
3,9. Wrapper Optimizado (Ridge),5,0.7767
4,6. Wrapper Backward,5,0.7553
5,5. Wrapper Forward,5,0.7428
6,10. Reducción Lineal PCA,4,0.6443
7,8. Embebido ElasticNet,4,0.6399
8,3. Filtro Univariado F-Test,5,0.6395
9,4. Filtro Univariado Info Mutua,5,0.6276


## 6.2

In [18]:
from collections import Counter
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
import pandas as pd


# 1. Juntamos a todas las ganadoras de los ejercicios 3 y 4 
todas_las_elegidas = (
    # list(X_final_ftest.columns) +   # Ej. 3: F-Test (5 var)
    list(X_final_mi.columns) +      # Ej. 3: Info Mutua (5 var)
    list(cols_backward) +           # Ej. 4: Wrapper Backward (5 var)
    list(cols_lasso)                # Ej. 4: Embebido Lasso (8 var)
)


conteo = Counter(todas_las_elegidas)

#extraemos solo los nombres de las 5 más votadas
top_5_nombres = [var for var, cantidad in conteo.most_common(5)]

print("LAS 5 VARIABLES AS VOTADAS:")
for var, cantidad in conteo.most_common(5):
    print(f" - {var}: {cantidad} votos")


#Signo e Importancia
# Entrenamos el Arbol solo con las 5 ganadoras para sacar la importancia
arbol = DecisionTreeRegressor(random_state=42)
arbol.fit(X_3_1[top_5_nombres], y)

# Entrenamos la Regresion Lineal solo con las 5 ganadoras para sacar el Signo
scaler = StandardScaler()
X_sc = scaler.fit_transform(X_3_1[top_5_nombres])

lineal = LinearRegression()
lineal.fit(X_sc, y)

#mostramos la tabla final
print("\n--- ANÁLISIS DEL EFECTO ---")
df_analisis = pd.DataFrame({
    'Variable': top_5_nombres,
    'Votos': [conteo[var] for var in top_5_nombres],
    'Importancia_Arbol': arbol.feature_importances_,
    'Signo_Lineal': lineal.coef_
})

print(df_analisis.to_string(index=False))


LAS 5 VARIABLES AS VOTADAS:
 - energy_use_kg_of_oil_equivalent_per_capita: 3 votos
 - ratio_auto_agriculture_value_added_of_gdp_div_government_integrity: 3 votos
 - life_expectancy_at_birth_total_years: 2 votos
 - government_integrity: 2 votos
 - government_integrity infant_mortality_rate_per_1_000_live_births: 2 votos

--- ANÁLISIS DEL EFECTO ---
                                                          Variable  Votos  Importancia_Arbol  Signo_Lineal
                        energy_use_kg_of_oil_equivalent_per_capita      3           0.053876   5361.770881
ratio_auto_agriculture_value_added_of_gdp_div_government_integrity      3           0.698781   3019.641979
                              life_expectancy_at_birth_total_years      2           0.169373    764.070810
                                              government_integrity      2           0.034643  12451.811722
  government_integrity infant_mortality_rate_per_1_000_live_births      2           0.043327  -2866.939555


## 6.3

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PowerTransformer
from sklearn.feature_selection import VarianceThreshold, SelectFromModel
from sklearn.linear_model import Ridge, LassoCV
from sklearn.metrics import r2_score, mean_squared_error



# 1. Separamos los datos partiendo del dataset de 65 variables (X_todas)
# IMPORTANTE: Asegúrate de reemplazar 'X_todas' por el nombre de la variable de las 65 columnas que tengas en tu notebook
X_train, X_test, y_train, y_test = train_test_split(X_todas, y, test_size=0.2, random_state=42)

# 2. Construimos el Súper-Pipeline End-to-End
pipeline_final = Pipeline([
    # Paso A: Limpieza de basura (Elimina variables constantes con Varianza 0)
    ('filtro_varianza', VarianceThreshold(threshold=0.0)),
    
    # Paso B: Transformación y Escalado (Normaliza curvas asimétricas y escala centrando en 0)
    ('preprocesamiento', PowerTransformer(method='yeo-johnson')),
    
    # Paso C: Selección de Variables INTELIGENTE 
    # Usamos LassoCV porque es ultra-rápido manejando 65 variables y elimina clones (correlación) automáticamente.
    ('seleccion', SelectFromModel(LassoCV(cv=5, random_state=42))),
    
    # Paso D: Modelo Predictivo Final
    ('modelo', Ridge(alpha=1.0))
])

# 3. Entrenamos el Pipeline COMPLETO
print("Entrenando el Pipeline Final... (paciencia, Lasso está podando las 65 variables)")
pipeline_final.fit(X_train, y_train)

# 4. Examen Final sobre datos inéditos
y_pred = pipeline_final.predict(X_test)

r2_final = r2_score(y_test, y_pred)
mse_final = mean_squared_error(y_test, y_pred)

print("\n--- EXAMEN FINAL SOBRE EL SET RESERVADO (DATOS INÉDITOS) ---")
print(f"R2 Score: {r2_final:.4f}")
print(f"MSE:      {mse_final:,.2f}")

# Extra: Ver cuántas variables sobrevivieron al proceso interno
columnas_vivas = pipeline_final.named_steps['seleccion'].get_support().sum()
print(f"\nVariables originales: {X_train.shape[1]}")
print(f"Variables que sobrevivieron al filtro interno y se usaron para predecir: {columnas_vivas}")
